# 30 · Manager + 3 Workers 多智能体协作

> **学习目标**：实现一个 Manager-Worker 多 Agent 系统 —— Manager 拆任务、并行派给 3 个专业化 Worker（搜资料 / 评审 / 写作）、汇总。理解多 Agent 何时**比单 Agent 强**、何时**反而拉垮**。
>
> **预备**：25–29 跑过；懂 asyncio.gather（[01_python_features.ipynb](../../../00-基础-Foundations/practice/stage1_入门/01_python_features.ipynb)）。
>
> **为什么重要**：Claude Code 的 sub-agent / CrewAI / AutoGen 都是这套架构。多 Agent 的**最大坑** = 容易拉垮（失败率叠加），先理解原理才能用对。

In [1]:
MODE = 'OFFLINE'

import asyncio, time, json, re, requests
from dataclasses import dataclass, field
from typing import Callable
OLLAMA = 'http://127.0.0.1:11434'
print(f'MODE = {MODE}')

MODE = OFFLINE


D:\ProgramData\anaconda3\envs\rag\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## 1. Manager-Worker 架构 —— 4 种多 Agent 协作模式之一

**4 种主流模式**（[Anthropic · Building effective agents](https://www.anthropic.com/research/building-effective-agents) 总结）：

| 模式 | 关键词 | 例子 |
|------|-------|------|
| **Manager-Worker**（本 notebook） | 一头一手，子任务派发 | Claude Code 的 sub-agent / 写综述 |
| **Pipeline** | 顺序串行 A→B→C | 翻译 → 校对 → 排版 |
| **Debate / Critique** | 互相 review | 答案+评审，多轮迭代 |
| **Mixture-of-Agents** | 多模型投票 | 同问题问 3 个 LLM 取众数 |

**Manager-Worker 关键特性**：
- Worker 之间**没有直接通信**（避免雪崩）
- Worker 输出汇到 Manager，由 Manager 决定如何合
- Worker 可**并行**执行（性能关键）

## 2. 定义 3 个专业化 Worker

**核心设计**：每个 Worker 只擅长一件事。**单一职责** 让 prompt 短、行为可预测。

In [2]:
# Worker 抽象：异步 callable，接受 task 文本，返回结果
@dataclass
class WorkerResult:
    worker: str
    task: str
    output: str
    latency_s: float
    ok: bool
    error: str = ''

async def run_worker(name: str, fn: Callable, task: str) -> WorkerResult:
    t0 = time.perf_counter()
    try:
        out = await fn(task)
        return WorkerResult(name, task, out, time.perf_counter() - t0, True)
    except Exception as e:
        return WorkerResult(name, task, '', time.perf_counter() - t0, False, str(e))

# === Worker 1: Researcher（搜资料） ===
async def researcher(task: str) -> str:
    await asyncio.sleep(0.3)   # 模拟 LLM + 工具延迟
    KB = {
        'rag':   '检索增强生成（RAG）通过检索外部知识库缓解 LLM 幻觉。流程：load→chunk→embed→retrieve→generate。',
        'agent': 'Agent 是能自主决定下一步动作的 LLM 系统。核心 = LLM + tools + loop。ReAct 是经典范式。',
        '比较':  'RAG 解决「知识广度」，Agent 解决「动作链」。生产里两者常组合：Agent 用 RAG 当一种工具。',
    }
    facts = [v for k, v in KB.items() if k in task.lower()]
    if not facts:
        return f'(researcher) 关于 "{task}" 没有现成资料。'
    return '【资料】\n' + '\n'.join(f'- {f}' for f in facts)

# === Worker 2: Critic（评审） ===
async def critic(task: str) -> str:
    await asyncio.sleep(0.2)
    issues = []
    if '幻觉' not in task: issues.append('未提及 RAG 解决的「幻觉」问题')
    if len(task) < 50:    issues.append('内容过短，建议展开')
    if 'Agent' in task and 'tool' not in task.lower(): issues.append('Agent 部分缺 tool use 概念')
    if not issues:
        return '【评审】通过。覆盖完整、表述清晰。'
    return '【评审】发现 ' + str(len(issues)) + ' 个问题:\n' + '\n'.join(f'  ⚠ {i}' for i in issues)

# === Worker 3: Writer（写作） ===
async def writer(task: str) -> str:
    await asyncio.sleep(0.4)
    # task 是 manager 拼好的「资料 + 评审」内容
    return f'【综述】\n基于研究员收集的资料与评审反馈，关于该主题：\n\n{task[:300]}...\n\n（综述就此结束）'

WORKERS = {'researcher': researcher, 'critic': critic, 'writer': writer}
print('已注册 3 个 worker:', list(WORKERS))

已注册 3 个 worker: ['researcher', 'critic', 'writer']


## 3. Manager —— 规划 + 派发 + 汇总

**3 步流程**：
1. **规划**：把 user 任务拆成「分给哪些 worker / 各自任务」的计划
2. **派发**：`asyncio.gather` 并行调 worker
3. **汇总**：把 worker 输出拼成最终答案（可能多轮：先 researcher + critic 并行 → 把结果给 writer）

In [3]:
@dataclass
class ManagerTrace:
    plan: list[dict] = field(default_factory=list)        # [{'worker','task'}]
    results: list[WorkerResult] = field(default_factory=list)
    rounds: int = 0
    total_s: float = 0.0

async def manager_run(user_task: str, verbose: bool = True) -> tuple[str, ManagerTrace]:
    """两阶段：
       Round 1: researcher + critic 并行（critic 评审 user_task 的覆盖面）
       Round 2: writer 单独跑，输入 = round 1 拼好的「资料 + 评审」
    """
    trace = ManagerTrace()
    t_start = time.perf_counter()

    # ---- 规划 ----
    plan = [
        {'worker': 'researcher', 'task': user_task},
        {'worker': 'critic',     'task': user_task},
    ]
    trace.plan.extend(plan)
    if verbose:
        print(f'📋 Manager 计划 round 1: {len(plan)} 个并行 worker')

    # ---- Round 1: 并行 ----
    trace.rounds += 1
    r1 = await asyncio.gather(*[run_worker(p['worker'], WORKERS[p['worker']], p['task']) for p in plan])
    trace.results.extend(r1)
    if verbose:
        for r in r1:
            print(f'  [{r.worker}] ok={r.ok} {r.latency_s:.2f}s  output={r.output[:60]!r}')

    # ---- Round 2: 串行（writer 需要 round1 输入） ----
    writer_input = '\n\n'.join(r.output for r in r1 if r.ok)
    trace.plan.append({'worker': 'writer', 'task': writer_input[:200]})
    trace.rounds += 1
    if verbose: print(f'\n📋 Manager 计划 round 2: writer 综合')
    r2 = await run_worker('writer', WORKERS['writer'], writer_input)
    trace.results.append(r2)
    if verbose: print(f'  [{r2.worker}] ok={r2.ok} {r2.latency_s:.2f}s')

    trace.total_s = time.perf_counter() - t_start
    return r2.output, trace

In [4]:
# 跑一个 task
answer, trace = await manager_run('请整理 RAG 和 Agent 的对比，给我一段综述', verbose=True)
print(f'\n{"="*60}\n📝 Final Answer:\n{answer}')
print(f'\n📊 总耗时: {trace.total_s:.2f}s （rounds={trace.rounds}, workers={len(trace.results)}）')
print(f'   理论串行耗时: {sum(r.latency_s for r in trace.results):.2f}s')
print(f'   实际并行节省: {sum(r.latency_s for r in trace.results) - trace.total_s:.2f}s')

📋 Manager 计划 round 1: 2 个并行 worker


  [researcher] ok=True 0.31s  output='【资料】\n- 检索增强生成（RAG）通过检索外部知识库缓解 LLM 幻觉。流程：load→chunk→embed→ret'
  [critic] ok=True 0.20s  output='【评审】发现 3 个问题:\n  ⚠ 未提及 RAG 解决的「幻觉」问题\n  ⚠ 内容过短，建议展开\n  ⚠ Agent '

📋 Manager 计划 round 2: writer 综合


  [writer] ok=True 0.41s

📝 Final Answer:
【综述】
基于研究员收集的资料与评审反馈，关于该主题：

【资料】
- 检索增强生成（RAG）通过检索外部知识库缓解 LLM 幻觉。流程：load→chunk→embed→retrieve→generate。
- Agent 是能自主决定下一步动作的 LLM 系统。核心 = LLM + tools + loop。ReAct 是经典范式。

【评审】发现 3 个问题:
  ⚠ 未提及 RAG 解决的「幻觉」问题
  ⚠ 内容过短，建议展开
  ⚠ Agent 部分缺 tool use 概念...

（综述就此结束）

📊 总耗时: 0.72s （rounds=2, workers=3）
   理论串行耗时: 0.92s
   实际并行节省: 0.20s


## 4. 并行 vs 串行 —— 量化收益

**并行只在 worker 之间无依赖时有效**。如果 worker B 需要 A 的输出 → 必须串行。

In [5]:
# 对比：串行版本
async def manager_serial(user_task: str) -> tuple[str, float]:
    t0 = time.perf_counter()
    r1 = await run_worker('researcher', researcher, user_task)
    r2 = await run_worker('critic',    critic,    user_task)
    r3 = await run_worker('writer',    writer,    r1.output + '\n\n' + r2.output)
    return r3.output, time.perf_counter() - t0

async def manager_parallel(user_task: str) -> tuple[str, float]:
    return await manager_run(user_task, verbose=False)[:2] if False else None

# 跑 5 次取平均
task = '帮我对比 RAG 和 Agent，写一段综述'
serial_times, parallel_times = [], []
for _ in range(3):
    _, ts = await manager_serial(task)
    serial_times.append(ts)
    _, trace = await manager_run(task, verbose=False)
    parallel_times.append(trace.total_s)

print(f'串行 3 次平均: {sum(serial_times)/3:.2f}s')
print(f'并行 3 次平均: {sum(parallel_times)/3:.2f}s')
print(f'节省比例:     {(1 - sum(parallel_times)/sum(serial_times))*100:.0f}%')
print('\n→ 并行收益取决于「最长依赖路径」。这里 round1 (max 0.3s) + round2 (0.4s) = 0.7s 是下限。')

串行 3 次平均: 0.92s
并行 3 次平均: 0.72s
节省比例:     22%

→ 并行收益取决于「最长依赖路径」。这里 round1 (max 0.3s) + round2 (0.4s) = 0.7s 是下限。


## 5. 多 Agent 的失败率叠加陷阱 —— 经典反模式

**关键直觉**：3 个 Worker 各有 90% 成功率，**总成功率 = 0.9³ ≈ 73%**。Worker 越多，全链路成功率掉得越快。

**防御 3 招**：
1. **Worker 之间错误隔离**：一个失败不让其它垮，**汇总时 graceful degrade**
2. **关键 Worker 加重试**：尤其依赖外部 API 的
3. **End-to-end eval set**：跑 20 题，看总成功率是不是真的 > 单 Worker

In [6]:
import random

# 模拟一个不稳定的 worker（30% 概率失败）
async def flaky_researcher(task: str) -> str:
    await asyncio.sleep(0.2)
    if random.random() < 0.3:
        raise RuntimeError('researcher: 网络超时')
    return '【资料】（mock fact about ' + task[:20] + '）'

WORKERS['researcher'] = flaky_researcher

# 跑 10 次，看总成功率
random.seed(0)
n_succ_total = n_succ_worker = 0
for _ in range(10):
    answer, trace = await manager_run('测试', verbose=False)
    r_workers_ok = all(r.ok for r in trace.results)
    if r_workers_ok: n_succ_worker += 1
    # 关键：即使某 worker 失败，writer 仍能用剩下的输入产出 → 端到端成功
    writer_result = trace.results[-1]
    if writer_result.ok: n_succ_total += 1

print(f'所有 worker 都成功的次数: {n_succ_worker}/10')
print(f'端到端有输出（即使某 worker 挂了）: {n_succ_total}/10')
print('→ 看到端到端成功率 > all-worker 成功率，是因为 writer 容忍了 researcher 失败。')
print('→ 这就是「graceful degrade」的工程价值。')

# 恢复
WORKERS['researcher'] = researcher

所有 worker 都成功的次数: 9/10
端到端有输出（即使某 worker 挂了）: 10/10
→ 看到端到端成功率 > all-worker 成功率，是因为 writer 容忍了 researcher 失败。
→ 这就是「graceful degrade」的工程价值。


## 6. 何时该用多 Agent，何时不该

**该用**：
- 任务可清晰分解为独立子任务（如「资料 + 评审」可并行）
- 不同子任务需要不同 prompt / 工具集 / 模型
- 单 prompt 装不下，分发能利用 token 预算

**不该用**：
- 子任务有强依赖（串行只能更慢，且失败率叠加）
- 单 Agent + Plan 已经能搞定（多 Agent 增加复杂度）
- 实时性要求高（多 Agent 延迟 ≥ 单 Agent）

**经验**：**先做单 Agent，做不下去再上多 Agent**。

## 深入思考

1. **Manager 自己就是个 Agent，怎么避免它「主 Agent 决策错」？**
   - 让 Manager 的 prompt **狭窄**：只负责「拆任务」「拼结果」，不要让它自由发挥。**复杂度集中在 worker，Manager 越笨越稳**。
2. **3 个 worker 的输出要不要再过一遍评审？**
   - 看场景。**高赌注**（如医疗 / 法律）= 应再加 critic 跑一遍 writer 输出 = Critique pattern。**普通场景**直接出。
3. **Worker 用什么模型？**
   - **不同任务用不同模型**才是多 Agent 的另一大价值。Researcher 用便宜模型 + 大 context；Critic 用 reasoning 模型；Writer 用风格强的模型。
4. **Claude Code 的 `Agent` 工具是怎么做隔离的？**
   - 子 Agent 跑在独立的 prompt context 里，**只回结果摘要给主 Agent**。主 Agent 上下文不被子任务大量信息污染。本 notebook 简化没体现，但思想一致。
5. **多 Agent 调试比单 Agent 难多少？**
   - 至少 N 倍（N = worker 数）。**最好早期就接 trace + 可视化**（见 [27 号 notebook](../stage1_入门/27_observe_claude_code.ipynb)），故障定位变成「看哪个 worker / 哪个 round 挂了」。

**改一改**：
- 加第 4 个 worker（如 `translator` 翻译综述到英文），看 round 数和延迟变化
- 把 researcher 改成 50% 失败率，看 graceful degrade 的极限在哪

## 自检 ✅

- [ ] 列出 4 种多 Agent 协作模式 + 各自适用场景
- [ ] 解释「Worker 失败率叠加」的数学（0.9³ ≈ 0.73）
- [ ] 解释「Manager 越笨越稳」是什么意思
- [ ] 给一个任务，能立刻判断「该用单 Agent vs 多 Agent」
- [ ] 默写 `asyncio.gather` 并行调用模式（3 行内）

## 下一步

→ [`31_state_machine_langgraph.ipynb`](31_state_machine_langgraph.ipynb)